# Support Vector Machines: From Convex Optimization to Gradient Descent

This notebook walks through the Support Vector Machine (SVM) classifier from two complementary perspectives: solving the constrained convex formulation with CVXPY, and solving the unconstrained hinge-loss formulation with gradient descent in PyTorch. You will see how the same problem can be approached via disciplined convex programming or first-order optimization with automatic differentiation, and understand the trade-offs of each approach.

**Learning objectives:**
- Formulate the hard-margin and soft-margin SVM as convex optimization problems.
- Solve the constrained SVM using CVXPY.
- Reformulate the SVM as an unconstrained problem using the hinge loss and solve it with gradient-based methods.
- Compare different optimizers (SGD, momentum, Adagrad, Adam) on the SVM objective.
- Extend to nonlinear decision boundaries using neural networks.

![SVM, see https://en.wikipedia.org/wiki/Support_vector_machine](https://upload.wikimedia.org/wikipedia/commons/thumb/7/72/SVM_margin.png/512px-SVM_margin.png)

## Background: Support Vector Machines

SVMs are a powerful supervised learning method for classification. The core idea is to find a hyperplane that separates two classes while maximizing the **margin** -- the distance between the hyperplane and the nearest data points from each class. These nearest points are called **support vectors**.

Given training data $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$ with $\mathbf{x}_i \in \mathbb{R}^d$ and $y_i \in \{-1, +1\}$, the decision boundary is the hyperplane $\{\mathbf{x} : \mathbf{w}^\top \mathbf{x} + b = 0\}$.

### Hard-Margin SVM

When the data are linearly separable, we can find a hyperplane that perfectly classifies all points. The hard-margin SVM finds the maximum-margin separator:

$$
\min_{\mathbf{w}, b} \;\frac{1}{2}\|\mathbf{w}\|^2 \quad \text{subject to} \quad y_i(\mathbf{w}^\top \mathbf{x}_i + b) \geq 1, \quad i = 1, \ldots, n.
$$

The margin width is $2 / \|\mathbf{w}\|$, so minimizing $\|\mathbf{w}\|^2$ maximizes the margin. This is a convex quadratic program (QP) with linear constraints.

### Soft-Margin SVM (Constrained Form)

Real data are rarely perfectly separable. The soft-margin SVM introduces slack variables $\zeta_i \geq 0$ to allow misclassifications, penalized by a parameter $C > 0$:

$$
\min_{\mathbf{w}, b, \boldsymbol{\zeta}} \;\frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^n \zeta_i \quad \text{subject to} \quad y_i(\mathbf{w}^\top \mathbf{x}_i + b) \geq 1 - \zeta_i, \quad \zeta_i \geq 0, \quad \forall\, i.
$$

- $C$ controls the trade-off: large $C$ penalizes misclassifications heavily (smaller margin, fewer errors), while small $C$ favors a wider margin at the expense of some errors.
- $\zeta_i > 0$ means point $i$ violates the margin; $\zeta_i > 1$ means it is misclassified.

This is still a convex QP, solvable by CVXPY or any QP solver.

### Soft-Margin SVM (Unconstrained Hinge-Loss Form)

By eliminating the slack variables, the constrained problem above is equivalent to the unconstrained regularized problem:

$$
\min_{\mathbf{w}, b} \;\frac{1}{2}\|\mathbf{w}\|^2 + C \cdot \frac{1}{n}\sum_{i=1}^n \underbrace{\max\bigl(0,\; 1 - y_i(\mathbf{w}^\top \mathbf{x}_i + b)\bigr)}_{\text{hinge loss}}.
$$

The hinge loss is convex and piecewise linear (hence subdifferentiable), which means we can apply gradient-based methods using automatic differentiation. After finding approximate minimizers $\widehat{\mathbf{w}}, \widehat{b}$, the classifier is $\mathbf{x} \mapsto \operatorname{sgn}(\widehat{\mathbf{w}}^\top \mathbf{x} + \widehat{b})$.

**Prerequisites:** This notebook requires `numpy`, `scikit-learn`, `matplotlib`, `cvxpy`, and `torch`. Install them via `pip install numpy scikit-learn matplotlib cvxpy torch` if they are not already available in your environment.

In [ ]:
import numpy as np
import sklearn
import matplotlib
import cvxpy as cp
import torch

from matplotlib import pyplot as plt
from matplotlib.colors import ListedColormap

np.random.seed(632)  # Fix seed for reproducibility

### Visualization Helpers

The following cell defines two plotting functions used throughout the notebook: `plot_binary` draws data points colored by class with an optional linear decision boundary and margin lines, and `plot_binary_with_model` does the same for an arbitrary PyTorch model by evaluating it on a dense grid.

In [ ]:
## Plotting utilities for binary classification

def get_line(W, b, x):
    """Given w and b, return x2 = (-w1*x1 + b) / w2 for the decision line."""
    return (-W[0]*x + b) / W[1]

def plot_binary(X, Y, W=None, b=None):
    """Plot 2D data with optional linear decision boundary and margin lines."""
    x1_min, x1_max = X[:, 0].min() - .5, X[:, 0].max() + .5
    x2_min, x2_max = X[:, 1].min() - .5, X[:, 1].max() + .5
    x1 = np.arange(x1_min, x1_max, .02)
    xx1, xx2 = np.meshgrid(x1, np.arange(x2_min, x2_max, .02))
    if W is not None and b is not None:
        # Evaluate w^T x + b on the grid for the filled contour
        decision = np.c_[xx1.ravel(), xx2.ravel()] @ W + b
        plt.contourf(xx1, xx2, decision.reshape(xx1.shape), cmap=plt.cm.RdBu, alpha=.8)
        plt.colorbar()
        # Draw decision boundary (intercept=0) and margin lines (intercept=+/-1)
        for intercept in [-1, 0, 1]:
            plt.plot(x1, get_line(W, b + intercept, x1), c='black')
    cm_bright = ListedColormap(['#FF0000', '#0000FF'])
    plt.scatter(X[:, 0], X[:, 1], c=Y.ravel(), cmap=cm_bright)
    plt.xlim(x1_min, x1_max)
    plt.ylim(x2_min, x2_max)
    plt.show()


def plot_binary_with_model(X, Y, model, threshold=0):
    """Plot 2D data with decision region from an arbitrary PyTorch model."""
    x1_min, x1_max = X[:, 0].min() - .5, X[:, 0].max() + .5
    x2_min, x2_max = X[:, 1].min() - .5, X[:, 1].max() + .5
    x1 = np.arange(x1_min, x1_max, .02)
    xx1, xx2 = np.meshgrid(x1, np.arange(x2_min, x2_max, .02))

    # Evaluate the model on a dense grid
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    grid_tensor = torch.tensor(grid, dtype=torch.float32)
    decision_values = model(grid_tensor).detach().numpy()

    decision_binary = np.sign(decision_values - threshold)
    plt.contourf(xx1, xx2, decision_binary.reshape(xx1.shape), cmap=plt.cm.RdBu, alpha=.8)
    plt.colorbar()

    cm_bright = ListedColormap(['#FF0000', '#0000FF'])
    plt.scatter(X[:, 0], X[:, 1], c=Y.ravel(), cmap=cm_bright)
    plt.xlim(x1_min, x1_max)
    plt.ylim(x2_min, x2_max)
    plt.show()

## Part 1: Linearly Separable Data

We generate a synthetic 2D dataset with two Gaussian clusters centered at $(-1, -1)$ and $(1, 1)$. The labels are set to $y_i \in \{-1, +1\}$. Because the clusters are well-separated (standard deviation 0.5), a linear SVM should achieve perfect or near-perfect classification.

In [ ]:
from sklearn.datasets import make_blobs

# Generate 700 points from two Gaussian clusters
X, Y = make_blobs(n_samples=700, centers=[(-1, -1), (1, 1)], cluster_std=0.5)
Y[Y == 0] = -1  # Convert labels from {0, 1} to {-1, +1}

plot_binary(X, Y)

### Solving the Soft-Margin SVM with CVXPY

[CVXPY](https://www.cvxpy.org/) is a Python-embedded modeling language for convex optimization. You declare decision variables, specify an objective and constraints using a natural mathematical syntax, and CVXPY verifies convexity (via Disciplined Convex Programming rules) before dispatching to a numerical solver.

Below we solve the constrained soft-margin SVM: minimize $\frac{1}{2}\|\mathbf{w}\| + C \sum_i \zeta_i$ subject to the margin and non-negativity constraints. CVXPY handles the QP reformulation and solver selection automatically.

In [ ]:
X = np.array(X)
Y = np.array(Y)

n = X.shape[0]  # Number of data points

# Decision variables
w = cp.Variable(X.shape[1])   # weight vector
b = cp.Variable()              # bias scalar
zeta = cp.Variable(n)          # slack variables (one per data point)

C = 10.0  # Regularization parameter

# Objective: minimize (1/2)||w|| + C * sum(zeta)
objective = cp.Minimize(cp.norm(w, 2) / 2 + C * cp.sum(zeta))

# Margin constraints: y_i(w^T x_i + b) >= 1 - zeta_i
constraints = [
    Y[i] * (cp.matmul(w, X[i]) + b) >= 1 - zeta[i] for i in range(n)
]
# Non-negativity of slack variables
constraints += [zeta[i] >= 0 for i in range(n)]

# Solve the QP
prob = cp.Problem(objective, constraints)
prob.solve()

w_opt = w.value
b_opt = b.value

print("Optimal w:", w_opt)
print("Optimal b:", b_opt)

plot_binary(X, Y, W=w_opt, b=b_opt)

### Solving the SVM with Gradient Descent (PyTorch)

Instead of solving the constrained QP, we now minimize the **unconstrained hinge-loss formulation**:

$$
L(\mathbf{w}, b) = \frac{1}{2}\|\mathbf{w}\|^2 + C \cdot \frac{1}{n}\sum_{i=1}^n \max\bigl(0,\; 1 - y_i(\mathbf{w}^\top \mathbf{x}_i + b)\bigr).
$$

PyTorch's automatic differentiation engine computes $\nabla_{\mathbf{w}} L$ and $\partial L / \partial b$ via backpropagation. We then update parameters with vanilla gradient descent: $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla_{\mathbf{w}} L$.

Setting `requires_grad=True` on a tensor tells PyTorch to track all operations on it. After calling `loss.backward()`, the `.grad` attribute holds the gradient.

First, let us visualize the hinge loss $\max(0, 1 - t)$ and its subgradient:

In [ ]:
from matplotlib import pyplot as plt

def hinge(t):
    """Hinge loss: max(0, 1-t)."""
    return torch.clamp(1 - t, min=0)

t = torch.linspace(-1, 2, 300)

# Compute the subgradient of the hinge loss via AD
subgrad_hinge = torch.autograd.functional.jacobian(hinge, t)

plt.plot(t, hinge(t))
plt.title("Hinge loss: max(0, 1-t)")
plt.xlabel("t")
plt.show()

plt.plot(t, subgrad_hinge)
plt.title("Subgradient of hinge loss")
plt.xlabel("t")
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt

# Convert data to PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
Y = torch.tensor(Y, dtype=torch.float32)

n_features = X.shape[1]
n_samples = X.shape[0]

# Initialize parameters with requires_grad=True for AD
w = torch.randn(n_features, requires_grad=True)
b = torch.randn(1, requires_grad=True)

# Hyperparameters
learning_rate = 0.01
C = 2
num_epochs = 100

def hinge_loss(y_pred, y_true):
    """Hinge loss: max(0, 1 - y_pred * y_true)."""
    return torch.clamp(1 - y_pred * y_true, min=0)

def softmargin_loss(w, b, X, Y):
    """Soft-margin SVM objective: (1/2)||w||^2 + C * mean(hinge losses)."""
    pred = X @ w + b
    hinge = hinge_loss(pred, Y).mean()
    return (w**2).sum() / 2.0 + C * hinge

# Training loop: vanilla gradient descent
losses = []
for epoch in range(num_epochs):
    loss = softmargin_loss(w, b, X, Y)  # Forward pass
    losses.append(loss.item())

    loss.backward()  # Backward pass: compute gradients

    with torch.no_grad():  # Disable gradient tracking during update
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        w.grad.zero_()  # Reset gradients for next iteration
        b.grad.zero_()

# Plot loss curve
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curve over Epochs')
plt.show()

w_final = w.detach().numpy()  # Detach from computational graph
b_final = b.detach().numpy()

print("Optimal w:", w_final)
print("Optimal b:", b_final)

plot_binary(X, Y, W=w_final, b=b_final)

### Gradient Descent with Momentum

Vanilla gradient descent can be slow when the loss landscape has high curvature in some directions. **Momentum** accelerates convergence by maintaining an exponentially weighted moving average of past gradients:

$$
\mathbf{v}_{t+1} = \mu \mathbf{v}_t + \eta \nabla L(\mathbf{w}_t), \qquad \mathbf{w}_{t+1} = \mathbf{w}_t - \mathbf{v}_{t+1},
$$

where $\mu \in [0, 1)$ is the momentum coefficient (typically 0.9). This dampens oscillations and builds up speed along consistent gradient directions.

In [ ]:
## Gradient descent with momentum (manual implementation)

X = torch.tensor(X, dtype=torch.float32)
Y = torch.tensor(Y, dtype=torch.float32)

n_features = X.shape[1]
n_samples = X.shape[0]

# Re-initialize parameters
w = torch.randn(n_features, requires_grad=True)
b = torch.randn(1, requires_grad=True)

learning_rate = 0.01
C = 1.0
num_epochs = 100
momentum = 0.9

# Initialize velocity terms to zero
w_momentum = torch.zeros_like(w)
b_momentum = torch.zeros_like(b)

losses = []
for epoch in range(num_epochs):
    loss = softmargin_loss(w, b, X, Y)
    losses.append(loss.item())

    loss.backward()

    with torch.no_grad():
        # Update velocity: v = mu * v + eta * grad
        w_momentum = momentum * w_momentum + learning_rate * w.grad
        b_momentum = momentum * b_momentum + learning_rate * b.grad

        # Update parameters: w = w - v
        w -= w_momentum
        b -= b_momentum

        w.grad.zero_()
        b.grad.zero_()

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curve over Epochs with Momentum')
plt.show()

w_final = w.detach().numpy()
b_final = b.detach().numpy()

print("Optimal w:", w_final)
print("Optimal b:", b_final)

plot_binary(X, Y, W=w_final, b=b_final)

### Using PyTorch's Built-in Optimizers

In practice, you rarely implement gradient updates by hand. PyTorch's `torch.optim` module ([docs](https://pytorch.org/docs/stable/optim.html)) provides a rich collection of optimizers including SGD, SGD with momentum, Adagrad, and Adam. Combined with `DataLoader` for mini-batch sampling, this gives a clean and efficient training pipeline.

Below we compare four optimizers on the same SVM hinge-loss objective using mini-batch gradient descent. We wrap the SVM parameters in an `nn.Module` so they integrate naturally with the optimizer API.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Wrap data in a TensorDataset for mini-batch iteration
X_tensor = torch.tensor(X, dtype=torch.float32)
Y_tensor = torch.tensor(Y, dtype=torch.float32)

dataset = TensorDataset(X_tensor, Y_tensor)
batch_size = 100
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# Redefine softmargin_loss for nn.Module interface
def softmargin_loss(model, X, Y, C):
    scores = model(X).squeeze()
    hinge = torch.clamp(1 - Y * scores, min=0)
    w = model.weight if hasattr(model, 'weight') else model.w
    return 0.5 * (w * w).sum() + C * hinge.mean()


# Linear SVM model as an nn.Module
class SVMModel(torch.nn.Module):
    def __init__(self, n_features):
        super(SVMModel, self).__init__()
        self.w = torch.nn.Parameter(torch.randn(n_features))
        self.b = torch.nn.Parameter(torch.randn(1))

    def forward(self, X):
        return X @ self.w + self.b


# Hyperparameters
learning_rate = 0.01
C = 1.0
num_epochs = 20

# Dictionary of optimizer factories for comparison
optimizers = {
    "SGD": lambda params: torch.optim.SGD(params, lr=learning_rate),
    "Momentum": lambda params: torch.optim.SGD(params, lr=learning_rate, momentum=0.9),
    "Adagrad": lambda params: torch.optim.Adagrad(params, lr=learning_rate * 10),
    "Adam": lambda params: torch.optim.Adam(params, lr=learning_rate)
}

# Re-create data loader with smaller batch size
X_tensor = torch.tensor(X, dtype=torch.float32)
Y_tensor = torch.tensor(Y, dtype=torch.float32)
dataset = TensorDataset(X_tensor, Y_tensor)
batch_size = 50
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Store per-iteration losses for each optimizer
all_losses = {}

for name, opt_func in optimizers.items():
    model = SVMModel(X_tensor.shape[1])
    optimizer = opt_func(model.parameters())

    losses = []
    for epoch in range(num_epochs):
        for X_batch, Y_batch in dataloader:
            loss = softmargin_loss(model, X_batch, Y_batch, C)
            losses.append(loss.item())

            optimizer.zero_grad()  # Clear previous gradients
            loss.backward()        # Compute gradients
            optimizer.step()       # Update parameters

    all_losses[name] = losses
    print(f"Optimizer: {name}")
    print(f"w: {model.w.data}")
    print(f"b: {model.b.data}\n")

# Compare loss curves across optimizers
plt.figure(figsize=(10, 6))
for name, losses in all_losses.items():
    plt.plot(losses, label=name)
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.title('Loss Curves for Different Optimizers')
plt.legend()
plt.show()

## Part 2: Nonlinearly Separable Data

Linear SVMs fail when the classes cannot be separated by a hyperplane. Below we generate a classic **concentric circles** dataset where the two classes form nested rings. No linear decision boundary can separate them, as we will verify by running the CVXPY soft-margin SVM (which finds the best linear separator, but it will perform poorly).

In [ ]:
from sklearn.datasets import make_circles

# Generate concentric circles: inner ring (class -1) and outer ring (class +1)
X, Y = make_circles(n_samples=500, noise=0.05)
Y[Y == 0] = -1  # Convert labels to {-1, +1}

plot_binary(X, Y)

### Linear SVM on Nonlinear Data (CVXPY)

As expected, the linear soft-margin SVM cannot find a good separator for concentric circles. The decision boundary passes through the middle of the data, misclassifying many points.

In [ ]:
X = np.array(X)
Y = np.array(Y)

n = X.shape[0]

# Same constrained soft-margin SVM as before
w = cp.Variable(X.shape[1])
b = cp.Variable()
zeta = cp.Variable(n)

C = 2.0

objective = cp.Minimize(cp.norm(w, 2) / 2 + C * cp.sum(zeta))
constraints = [
    Y[i] * (cp.matmul(w, X[i]) + b) >= 1 - zeta[i] for i in range(n)
]
constraints += [zeta[i] >= 0 for i in range(n)]

prob = cp.Problem(objective, constraints)
prob.solve()

w_opt = w.value
b_opt = b.value

print("Optimal w:", w_opt)
print("Optimal b:", b_opt)

# The linear boundary clearly fails on this nonlinear data
plot_binary(X, Y, W=w_opt, b=b_opt)

### Nonlinear Decision Boundaries with a Neural Network

To handle nonlinear data, we replace the linear model $\mathbf{w}^\top \mathbf{x} + b$ with a two-hidden-layer ReLU network. The network learns a nonlinear feature map $\phi(\mathbf{x})$ such that the data become separable in the learned representation. We keep the same hinge-loss objective but now regularize all network weights.

Note that this optimization problem is **non-convex** (due to the neural network), so different optimizers may converge to different local minima. We compare SGD, momentum, Adagrad, and Adam below.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class TwoLayerReLUModel(nn.Module):
    """Two hidden-layer ReLU network for nonlinear classification."""
    def __init__(self, n_features, hidden_size):
        super(TwoLayerReLUModel, self).__init__()
        self.fc1 = nn.Linear(n_features, hidden_size)   # Input -> Hidden 1
        self.fc2 = nn.Linear(hidden_size, hidden_size)   # Hidden 1 -> Hidden 2
        self.fc3 = nn.Linear(hidden_size, 1)              # Hidden 2 -> Output

    def forward(self, X):
        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        return self.fc3(X)

# Redefine softmargin_loss for nn.Module (regularizes all parameters)
def softmargin_loss(model, X, Y, C):
    """Hinge loss + weight decay regularization for an nn.Module."""
    pred = model(X).squeeze()
    hinge = torch.clamp(1 - pred * Y, min=0).mean()
    # L2 regularization over all model parameters
    reg = sum(torch.sum(weight ** 2) for weight in model.parameters())
    return hinge + C * reg / 2.0


hidden_size = 100

# Prepare data
X_tensor = torch.tensor(X, dtype=torch.float32)
Y_tensor = torch.tensor(Y, dtype=torch.float32)
dataset = TensorDataset(X_tensor, Y_tensor)
batch_size = 50
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Hyperparameters
learning_rate = 0.01
C = 0.01       # Small regularization weight (network has many parameters)
num_epochs = 200

optimizers = {
    "SGD": lambda params: torch.optim.SGD(params, lr=learning_rate),
    "Momentum": lambda params: torch.optim.SGD(params, lr=learning_rate, momentum=0.9),
    "Adagrad": lambda params: torch.optim.Adagrad(params, lr=learning_rate * 10),
    "Adam": lambda params: torch.optim.Adam(params, lr=learning_rate)
}

all_losses = {}

for name, opt_func in optimizers.items():
    model = TwoLayerReLUModel(X_tensor.shape[1], hidden_size)
    optimizer = opt_func(model.parameters())

    losses = []
    for epoch in range(num_epochs):
        for X_batch, Y_batch in dataloader:
            loss = softmargin_loss(model, X_batch, Y_batch, C)
            losses.append(loss.item())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    all_losses[name] = losses
    plot_binary_with_model(X, Y, model)  # Show decision region for each optimizer

# Compare loss curves
plt.figure(figsize=(10, 6))
for name, losses in all_losses.items():
    plt.plot(losses, label=name)
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.title('Loss Curves for Different Optimizers')
plt.legend()
plt.show()

## Summary

**Key takeaways from this notebook:**

1. **Two formulations, one problem.** The soft-margin SVM can be expressed as a constrained QP (solved exactly by CVXPY) or as an unconstrained hinge-loss minimization (solved approximately by gradient descent). Both yield comparable decision boundaries on linearly separable data.

2. **CVXPY vs. gradient descent.** CVXPY is declarative and guarantees a global optimum for convex problems, but does not scale easily to very large datasets. Gradient-based methods scale well (especially with mini-batches) and extend naturally to non-convex models like neural networks.

3. **Optimizer choice matters.** Momentum accelerates convergence by smoothing gradient oscillations. Adaptive methods (Adagrad, Adam) automatically adjust per-parameter learning rates and often converge faster in practice, particularly for non-convex objectives.

4. **Nonlinear data requires nonlinear models.** A linear SVM cannot separate concentric circles. Replacing the linear model with a neural network allows learning nonlinear decision boundaries, at the cost of non-convexity (no global optimality guarantee).